In [1]:
import numpy as np
import pandas
import sklearn 
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn import datasets
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
data = pandas.read_csv('./Project.csv', delimiter=',')

In [3]:
data

,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low
...,...,...,...,...,...,...,...,...,...,...,...
1795,180,Weekend,6.33,8.16,0,4,0,5.59,73.5,31.91,Low
1796,180,Weekend,4.70,7.88,0,4,0,6.69,89.8,26.30,Low
1797,180,Weekend,3.92,6.39,2,1,0,6.77,74.6,34.07,Low
1798,180,Weekday,8.93,11.11,2,5,0,8.28,74.6,38.14,Low


In [4]:
hours_worked = data["work_hours"]
screen_time = data["screen_time_hours"]
meetings_count = data["meetings_count"]
breaks_taken = data["breaks_taken"]
after_hours_work = data["after_hours_work"]
sleep_hours = data["sleep_hours"]
task_completion = data["task_completion_rate"]
burnout_score = data["burnout_score"]
burnout_risk = data["burnout_risk"]

In [5]:
x = data.drop(columns = ['user_id','burnout_score'])
y = data['burnout_score']

In [6]:
x_nominal = pandas.get_dummies(x)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=500)
xn_train, xn_test, yn_train, yn_test = train_test_split(x_nominal, y, test_size=500)

labels = ['model'] + x_nominal.columns.to_list()
labels
spearman_table = pandas.DataFrame(columns = labels)
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium


## Linear

### Linear Regression

In [8]:
linear = LinearRegression()

In [9]:
linear.fit(xn_train, yn_train)
y_predicted_lr = linear.predict(xn_test)
mean_squared_error(yn_test, y_predicted_lr)

31.031995163674885

In [10]:
y_predicted_lr = pandas.DataFrame(y_predicted_lr)

In [11]:
vals = {}
vals['model'] = 'linear regression'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_lr, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

C:\Users\Adrian_Cosar\AppData\Local\Temp\ipykernel_31824\426421780.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])


## Decision Tree

### Regression Tree

In [12]:
tree = DecisionTreeRegressor()
parameters_tree = {# 'criterion':('squared_error','friedman_mse','absolute_error','poisson'),
                   'max_depth':[5,10,15,20],
                   'min_samples_split':[5,10,15,20],
                   'max_leaf_nodes':[5,10,20,50,None]
                  }
clf = GridSearchCV(tree, parameters_tree)
clf.fit(xn_train, yn_train)

,estimator,DecisionTreeRegressor()
,param_grid,"{'max_depth': [5, 10, ...], 'max_leaf_nodes': [5, 10, ...], 'min_samples_split': [5, 10, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [13]:
y_predicted_rt = pandas.DataFrame(clf.predict(xn_test))

vals = {}
vals['model'] = 'regression tree'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_rt, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

In [14]:
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.009676,0.010760,0.001780,-0.025570,0.083609,-0.012114,0.033412,0.014065,-0.014065,0.033299,-0.002854,-0.009410
0,regression tree,0.008368,0.010743,0.004411,-0.025185,0.082900,-0.018159,0.035976,0.011992,-0.011992,0.040656,-0.000483,-0.014632


In [15]:
x_nominal.columns

Index(['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
       'after_hours_work', 'sleep_hours', 'task_completion_rate',
       'day_type_Weekday', 'day_type_Weekend', 'burnout_risk_High',
       'burnout_risk_Low', 'burnout_risk_Medium'],
      dtype='object')

### Random Forest

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf_reg.fit(xn_train, yn_train)
y_pred_rf = pandas.DataFrame(rf_reg.predict(xn_test))

vals = {}
vals['model'] = 'random forest'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_rf, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

## Boosting

### Gradient Boosting

In [ ]:
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
gb_reg.fit(xn_train, yn_train)
y_pred_gb = pandas.DataFrame(gb_reg.predict(xn_test))

vals = {}
vals['model'] = 'gradient boosting'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_gb, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

In [18]:
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])
spearman_table


,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.009676,0.010760,0.001780,-0.025570,0.083609,-0.012114,0.033412,0.014065,-0.014065,0.033299,-0.002854,-0.009410
0,regression tree,0.008368,0.010743,0.004411,-0.025185,0.082900,-0.018159,0.035976,0.011992,-0.011992,0.040656,-0.000483,-0.014632
0,random forest,0.004457,0.006664,-0.001403,-0.024002,0.089870,-0.008499,0.029667,0.014204,-0.014204,0.037259,0.000000,-0.013873
0,gradient boosting,0.011399,0.013115,0.004654,-0.022686,0.083737,-0.013446,0.034183,0.014481,-0.014481,0.036217,-0.000037,-0.013446
0,gradient boosting,0.011399,0.013115,0.004654,-0.022686,0.083737,-0.013446,0.034183,0.014481,-0.014481,0.036217,-0.000037,-0.013446
